# 02 — Human Review Queue Overview

이 노트북은 후보 데이터의 사람 검토 진행률을 읽기 전용으로 요약합니다. 의미적 검토는 체크 박스를 채우는 형식적 절차가 아니라 실제 텍스트와 코드의 동작·주제·요구 작업이 일치하는지를 확인하는 것입니다.


In [1]:
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except ImportError:
    pass


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('/content/drive/MyDrive/PhysLLM_N02_Antigravity_Workspace')
if not PROJECT_ROOT.exists():
    candidate = Path.cwd().resolve()
    if (candidate / 'data/review/human_review_queue.csv').exists():
        PROJECT_ROOT = candidate
    elif (candidate.parent / 'data/review/human_review_queue.csv').exists():
        PROJECT_ROOT = candidate.parent

LEDGER = PROJECT_ROOT / 'data/review/human_review_queue.csv'
if not LEDGER.exists():
    raise FileNotFoundError(f'Review ledger not found: {LEDGER}. Run the workspace audit first or copy the workspace into Drive.')
review = pd.read_csv(LEDGER, dtype={'text_id': str})
print('Rows in queue:', len(review))
print('Status counts:')
display(review['review_status'].fillna('NOT_REVIEWED').value_counts(dropna=False).to_frame('rows'))
print('By style:')
display(pd.crosstab(review['style'], review['review_status'].fillna('NOT_REVIEWED')))


## Review checklist per row

1. The domain is correct and sufficiently specific for the assigned `topic_id`.
2. The intended operation in `operation_id` matches what the code actually computes.
3. Edge cases are addressed (empty input, missing values, equal min/max, short series, ties as relevant).
4. Text around the code describes the actual behavior and does not introduce a false scientific implication.
5. Content is not merely a number-swapped duplicate of another item unless that pairing is deliberate and documented.
6. The row is still excluded from pilot data and is not copied from a calibration prompt.

For approval, record reviewer, rationale, and review timestamp. If rejected, explain the required repair and preserve the original version.
